# Qualitative Diagnostic Analysis: Identity Performance, Inter-Identity Confusion, & Gallery Sensitivity

This notebook investigates the intrinsic visual factors, recording conditions, and retrieval error patterns that govern dog re-identification in **DogReID-1553**.

### Core Experiments:
1. **Best- vs. Worst-Performing Identity Analysis**:
   - Compute per-identity mAP across all 680 closed-world test identities.
   - Select the 10 best-performing and 10 worst-performing identities.
   - Extract representative query and gallery video crops for each selected identity.
   - Characterize each identity across **Dataset representation** (query/gallery support), **Visual Appearance** (coat pattern, colour, breed/type, body size, distinctive marks), and **Recording conditions** (video quality, viewpoint variation, visibility, occlusion).
   - Determine whether failure modes are primarily associated with visually ambiguous dogs or challenging recording conditions.

2. **Inter-Identity Similarity and Confusion Analysis**:
   - Analyze incorrect top-ranked predictions across all query tracklets.
   - Distinguish between **Unidirectional Confusion** ($A \to B$) and **Mutual Confusion** ($A \leftrightarrow B$, where Dog $A$ retrieves Dog $B$ AND Dog $B$ retrieves Dog $A$).
   - Identify the 10–20 most frequently confused identity pairs and compare their visual characteristics.
   - Present representative retrieval triplets: *(Query Dog A, False Top-1 Match Dog B, True Gallery Match Dog A)*.

3. **Gallery Size Sensitivity Analysis (Transferred & Enhanced)**:
   - **Volume Scaling**: Retrieval degradation across total gallery video counts (38.9% / 777 videos up to 100% / 1,998 videos) with 100-iteration bootstrap confidence intervals.
   - **Identity Scaling**: Retrieval degradation as the number of competing distractor identities in the gallery increases from 100 to 777 unique dogs.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pickle
from collections import defaultdict, Counter
from IPython.display import display

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

FIG_DIR = ROOT_DIR / "evaluation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")
print(f"Figure output directory: {FIG_DIR}")

# Set consistent matplotlib publication style
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.titlesize': 16,
    'font.family': 'sans-serif',
})

In [ ]:
# =====================================================================
# Bounding Box Index & High-Precision Image Cropping Helper
# =====================================================================
BBOX_CSV = ROOT_DIR / "bounding_boxes.csv"
bbox_df = pd.read_csv(BBOX_CSV)
bbox_map = {
    (str(r["DOG_ID"]), str(r["VIDEO_ID"])): (
        int(r["x_top_left"]), int(r["y_top_left"]), int(r["width"]), int(r["height"])
    )
    for _, r in bbox_df.iterrows()
}
print(f"Loaded {len(bbox_map)} ground truth bounding boxes from {BBOX_CSV.name}.")

def load_dog_crop(dog_id, video_id=None, pad=0.08):
    """Load and crop dog frame using ground-truth bounding box with padding."""
    dog_dir = ROOT_DIR / "Images" / str(dog_id)
    if not dog_dir.exists():
        return None
    if video_id is None:
        files = sorted(list(dog_dir.glob("*.jpg")))
        if not files: return None
        img_path = files[0]
        v_id = img_path.stem[37:]  # UUID is 36 chars + 1 hyphen
    else:
        img_path = dog_dir / f"{dog_id}-{video_id}.jpg"
        v_id = str(video_id)
        if not img_path.exists():
            files = sorted(list(dog_dir.glob("*.jpg")))
            if not files: return None
            img_path = files[0]
            v_id = img_path.stem[37:]
    try:
        im = Image.open(img_path).convert("RGB")
        box = bbox_map.get((str(dog_id), str(v_id)))
        if box:
            x, y, w, h = box
            x1 = max(0, int(x - w * pad))
            y1 = max(0, int(y - h * pad))
            x2 = min(im.width, int(x + w * (1.0 + pad)))
            y2 = min(im.height, int(y + h * (1.0 + pad)))
            if x2 > x1 and y2 > y1:
                im = im.crop((x1, y1, x2, y2))
        return im
    except Exception as e:
        return None

Loaded 7463 ground truth bounding boxes from bounding_boxes.csv.


## 1. Best- vs. Worst-Performing Identity Analysis

**Objective**: Investigate which intrinsic visual characteristics (coat patterns, breed, morphology) and recording conditions (viewpoint, distance, illumination, occlusions) are associated with easier vs. more difficult dog re-identification.

### Procedure:
1. Calculate per-identity mAP for all test identities from the closed-world baseline distance matrix.
2. Select the **10 best-performing** and **10 worst-performing** identities (filtering for identities with multiple queries for robust ranking).
3. Collect and visualize representative query and gallery crops for each identity.
4. Provide structured attribute annotations across **Dataset representation**, **Appearance**, and **Recording conditions**.

In [ ]:
# =====================================================================
# Compute Per-Identity mAP & Rank-1 Accuracy
# =====================================================================
baseline_csv = ROOT_DIR / "evaluation" / "csvs_closed_results" / "baseline_vid2vid" / "closed_dist_matrix.csv"
assert baseline_csv.exists(), f"Baseline CSV missing at {baseline_csv}"

df_full = pd.read_csv(baseline_csv)
get_id = lambda x: str(x).rsplit('_', 1)[0]
get_vid = lambda x: str(x).rsplit('_', 1)[1] if '_' in str(x) else str(x)

query_ids = np.array(df_full['queryId'].values)
gallery_ids = np.array(df_full.columns[1:].values)

query_labels = np.array([get_id(i) for i in query_ids])
gallery_labels = np.array([get_id(i) for i in gallery_ids])
dist_mat = df_full.iloc[:, 1:].values.astype(np.float32)

print(f"Loaded closed-world distance matrix: {len(query_labels)} queries x {len(gallery_labels)} gallery tracks.")
print(f"Unique query identities: {len(np.unique(query_labels))} | Unique gallery identities: {len(np.unique(gallery_labels))}")

# Pairwise match matrix & sorted indices
match_matrix = query_labels[:, None] == gallery_labels[None, :]
sort_idx = np.argsort(dist_mat, axis=1)
sorted_matches = np.take_along_axis(match_matrix, sort_idx, axis=1)

# Calculate per-query Average Precision (AP) and Rank-1 hit
ap_list = []
for i in range(len(query_labels)):
    row_m = sorted_matches[i]
    if not np.any(row_m):
        ap_list.append(0.0)
        continue
    cum_m = np.cumsum(row_m)
    rank_pos = np.where(row_m)[0] + 1.0
    precision = cum_m[row_m] / rank_pos
    ap_list.append(float(np.mean(precision)))

df_query_stats = pd.DataFrame({
    'query_id': query_ids,
    'dog_id': query_labels,
    'video_id': [get_vid(i) for i in query_ids],
    'AP': ap_list,
    'rank1': sorted_matches[:, 0].astype(int)
})

# Aggregate per-identity statistics
id_stats = df_query_stats.groupby('dog_id').agg(
    n_queries=('AP', 'count'),
    mAP=('AP', 'mean'),
    rank1_acc=('rank1', 'mean')
).reset_index()

gallery_counts = pd.Series(gallery_labels).value_counts()
id_stats['n_gallery'] = id_stats['dog_id'].map(gallery_counts).fillna(0).astype(int)
id_stats['mAP_pct'] = id_stats['mAP'] * 100.0
id_stats['rank1_pct'] = id_stats['rank1_acc'] * 100.0

print(f"\nSummary across all {len(id_stats)} identities:")
print(f"  Mean mAP          : {id_stats['mAP_pct'].mean():.2f}%")
print(f"  Median mAP        : {id_stats['mAP_pct'].median():.2f}%")
print(f"  Identities @ 100% : {sum(id_stats['mAP'] == 1.0)} ({sum(id_stats['mAP'] == 1.0)/len(id_stats)*100:.1f}%)")
print(f"  Identities @ 0%   : {sum(id_stats['rank1_acc'] == 0.0)} ({sum(id_stats['rank1_acc'] == 0.0)/len(id_stats)*100:.1f}%)")

Loaded closed-world distance matrix: 430 queries x 1998 gallery tracks.
Unique query identities: 179 | Unique gallery identities: 777

Summary across all 179 identities:
  Mean mAP          : 59.95%
  Median mAP        : 62.70%
  Identities @ 100% : 27 (15.1%)
  Identities @ 0%   : 34 (19.0%)


In [ ]:
# =====================================================================
# Selection of 10 Best- and 10 Worst-Performing Identities
# =====================================================================
# Best 10: 10 random identities selected from all identities with 100% mAP (reproducible seed=42)
map100_identities = id_stats[id_stats["mAP"] == 1.0].copy()
best_10_df = map100_identities.sample(n=10, random_state=42).copy()
best_10_df["Group"] = "Best 10 (Random 100% mAP)"

# Worst 10: Lowest mAP among identities with >=2 query samples (genuine hard cases)
worst_10_df = id_stats[id_stats["n_queries"] >= 2].sort_values(
    by=["mAP", "rank1_acc", "n_queries", "n_gallery"],
    ascending=[True, True, False, False]
).head(10).copy()
worst_10_df["Group"] = "Worst 10"

# Combine for clean summary
selected_20 = pd.concat([best_10_df, worst_10_df]).reset_index(drop=True)
selected_20["Short ID"] = selected_20["dog_id"].apply(lambda x: x[:8] + "...")

display_cols = ["Group", "Short ID", "mAP_pct", "rank1_pct", "n_queries", "n_gallery", "dog_id"]
display(selected_20[display_cols].rename(columns={
    "mAP_pct": "mAP (%)",
    "rank1_pct": "Rank-1 (%)",
    "n_queries": "N_q",
    "n_gallery": "N_g",
    "dog_id": "Full DOG_ID"
}))


,Group,Short ID,mAP (%),Rank-1 (%),N_q,N_g,Full DOG_ID
0,Best 10 (Random 100% mAP),12a6926f...,100.000000,100.0,3,2,12a6926f-d6c3-4654-9a02-550fbe277848
1,Best 10 (Random 100% mAP),1c2274d7...,100.000000,100.0,1,1,1c2274d7-87d8-432e-9022-b514af7aa483
2,Best 10 (Random 100% mAP),1747ddc7...,100.000000,100.0,1,4,1747ddc7-82e8-4f54-9699-1bee70811f3b
3,Best 10 (Random 100% mAP),2edc859f...,100.000000,100.0,2,3,2edc859f-b65a-4998-a3cb-403ac049d4eb
4,Best 10 (Random 100% mAP),01298aba...,100.000000,100.0,4,1,01298aba-8d74-417d-9ecb-c4fb0ea31425
5,Best 10 (Random 100% mAP),1b0206e8...,100.000000,100.0,1,1,1b0206e8-c12b-43fd-9eb0-32683b1405a7
6,Best 10 (Random 100% mAP),27ccedf7...,100.000000,100.0,3,2,27ccedf7-26c3-416b-b061-fc541c8370ab
7,Best 10 (Random 100% mAP),292fd429...,100.000000,100.0,2,2,292fd429-17e9-4981-a2d2-13fbe7f35913
8,Best 10 (Random 100% mAP),1bc9e4bc...,100.000000,100.0,2,2,1bc9e4bc-b049-4102-bc51-f9afc80af6ec
9,Best 10 (Random 100% mAP),36cdd3e8...,100.000000,100.0,2,3,36cdd3e8-0f49-478b-85eb-bbf1c865d0f3


In [ ]:
# =====================================================================
# Structured Attribute Annotation Table: Best 10 & Worst 10 Identities
# (Target CSV: evaluation/best_vs_worst_annotations.csv)
# =====================================================================
ANNOT_CSV = ROOT_DIR / "evaluation" / "best_vs_worst_annotations.csv"

EXPECTED_COLS = [
    "Group", "Short ID", "mAP (%)", "Rank-1 (%)", "N_q", "N_g", "Full DOG_ID",
    "Distinctive visual features", "Occlusion", "Blurriness / low resolution",
    "Lighting", "Viewpoint variation"
]

if ANNOT_CSV.exists():
    df_annot = pd.read_csv(ANNOT_CSV)
    for col in EXPECTED_COLS:
        if col not in df_annot.columns:
            df_annot[col] = ""
    df_annot = df_annot[EXPECTED_COLS]
else:
    df_annot = pd.DataFrame({
        "Group": selected_20["Group"].values,
        "Short ID": selected_20["Short ID"].values,
        "mAP (%)": selected_20["mAP_pct"].round(1).values,
        "Rank-1 (%)": selected_20["rank1_pct"].round(1).values,
        "N_q": selected_20["n_queries"].values,
        "N_g": selected_20["n_gallery"].values,
        "Full DOG_ID": selected_20["dog_id"].values,
        "Distinctive visual features": ["" for _ in range(20)],
        "Occlusion": ["" for _ in range(20)],
        "Blurriness / low resolution": ["" for _ in range(20)],
        "Lighting": ["" for _ in range(20)],
        "Viewpoint variation": ["" for _ in range(20)],
    })
    df_annot.to_csv(ANNOT_CSV, index=False)

print(f"Loaded {len(df_annot)} target identities for visual attribute annotation.")
print(f"CSV location: {ANNOT_CSV}")
display(df_annot)


Loaded 20 target identities for visual attribute annotation.
CSV location: c:\Users\marko\Desktop\DogReID-1553\evaluation\best_vs_worst_annotations.csv


,Group,Short ID,mAP (%),Rank-1 (%),N_q,N_g,Full DOG_ID,Distinctive visual features,Occlusion,Blurriness / low resolution,Lighting,Viewpoint variation
0,Best 10 (Random 100% mAP),a6c4d7b5...,100.0,100.0,1,4,a6c4d7b5-08ff-4530-8b51-c1470cbe35ac,NaN,NaN,NaN,NaN,NaN
1,Best 10 (Random 100% mAP),1985e9b0...,100.0,100.0,2,3,1985e9b0-d386-4cd3-ac64-4fc0621dd795,NaN,NaN,NaN,NaN,NaN
2,Best 10 (Random 100% mAP),05468985...,100.0,100.0,3,2,05468985-0eb6-4493-8cad-970e51ca9128,NaN,NaN,NaN,NaN,NaN
3,Best 10 (Random 100% mAP),ba5d57d7...,100.0,100.0,3,1,ba5d57d7-6916-456a-8707-83c570a63394,NaN,NaN,NaN,NaN,NaN
4,Best 10 (Random 100% mAP),7d894dd7...,100.0,100.0,4,1,7d894dd7-49db-41a1-add2-fa3dd116d651,NaN,NaN,NaN,NaN,NaN
5,Best 10 (Random 100% mAP),88648a2a...,100.0,100.0,3,2,88648a2a-8b4d-4ee0-b363-0b9a3724a06b,NaN,NaN,NaN,NaN,NaN
6,Best 10 (Random 100% mAP),40bf12ea...,100.0,100.0,1,1,40bf12ea-2b58-4e31-83d4-df87223b73bB,NaN,NaN,NaN,NaN,NaN
7,Best 10 (Random 100% mAP),59b463ad...,100.0,100.0,2,2,59b463ad-d64c-489a-b0d5-27647cc68e44,NaN,NaN,NaN,NaN,NaN
8,Best 10 (Random 100% mAP),d43d5070...,100.0,100.0,3,2,d43d5070-2998-4f32-997c-b3756f7c72dc,NaN,NaN,NaN,NaN,NaN
9,Best 10 (Random 100% mAP),1b0206e8...,100.0,100.0,1,1,1b0206e8-c12b-43fd-9eb0-32683b1405a7,NaN,NaN,NaN,NaN,NaN


### Interactive Annotation Tool: Best 10 vs. Worst 10 Identities

Use the interactive widget below to manually review and annotate each of the 20 target dogs across the **5 defined visual attributes**:
1. **Distinctive visual features** (`Yes` / `No`): Distinctive coat patterns, facial markings, unusual ears, tail, etc.
2. **Occlusion** (`Yes` / `No`): Dog is partially obscured in one or more relevant videos.
3. **Blurriness / low resolution** (`Yes` / `No`): Dog's appearance is noticeably blurred or lacks sufficient detail.
4. **Lighting** (`Good` / `Poor`): Sufficient lighting versus underexposure, overexposure or difficult lighting.
5. **Viewpoint variation** (`Low` / `High`): Limited viewpoint changes versus substantial changes (front, side, rear).

> **Auto-Save**: Changes are automatically written directly to `evaluation/best_vs_worst_annotations.csv` on selection.
>
> **Alternative Web Annotator**: You can also run `python evaluation/annotate_dogs.py --port 8050` or open `evaluation/best_vs_worst_annotator.html` in your browser for a standalone full-screen annotator!


In [9]:
# =====================================================================
# Interactive ipywidgets Tool for Manual Identity Annotation
# =====================================================================
import io
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

ATTR_CONFIG = [
    {
        "id": "Distinctive visual features",
        "label": "Distinctive features",
        "options": ["(Unset)", "Yes", "No"],
        "desc": "Distinctive coat patterns, facial markings, unusual ears, tail, etc.",
    },
    {
        "id": "Occlusion",
        "label": "Occlusion",
        "options": ["(Unset)", "Yes", "No"],
        "desc": "Dog is partially obscured in one or more relevant videos.",
    },
    {
        "id": "Blurriness / low resolution",
        "label": "Blurriness / low res",
        "options": ["(Unset)", "Yes", "No"],
        "desc": "Dog's appearance is noticeably blurred or lacks sufficient detail.",
    },
    {
        "id": "Lighting",
        "label": "Lighting",
        "options": ["(Unset)", "Good", "Poor"],
        "desc": "Sufficient lighting versus underexposure, overexposure or difficult lighting.",
    },
    {
        "id": "Viewpoint variation",
        "label": "Viewpoint variation",
        "options": ["(Unset)", "Low", "High"],
        "desc": "Limited viewpoint changes versus substantial changes (front, side, rear).",
    },
]

SPLITS_CSV = ROOT_DIR / "splits.csv"
splits = pd.read_csv(SPLITS_CSV)

current_idx = 0

def pil_to_widget_img(pil_img, max_w=170, max_h=170):
    if pil_img is None:
        return widgets.HTML("<div style='width:160px;height:120px;border:1px dashed #ccc;display:flex;align-items:center;justify-content:center;color:#888;'>No Image</div>")
    im = pil_img.copy()
    im.thumbnail((max_w, max_h))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    return widgets.Image(value=buf.getvalue(), format="jpeg", width=str(max_w))

def is_dog_annotated(idx):
    row = df_annot.iloc[idx]
    for attr in ATTR_CONFIG:
        v = row[attr["id"]]
        if pd.isna(v) or str(v).strip() in ("", "(Unset)"):
            return False
    return True

def make_selector_label(idx):
    r = df_annot.iloc[idx]
    done = "✓" if is_dog_annotated(idx) else "…"
    grp = "Best" if "Best" in str(r["Group"]) else "Worst"
    return f"[{idx+1:02d}/20] {grp} | {r['Short ID']} (mAP {r['mAP (%)']:.1f}%) [{done}]"

def save_csv():
    df_annot.to_csv(ANNOT_CSV, index=False)
    status_label.value = f"<span style='color:green;font-weight:bold;'>✓ Saved to {ANNOT_CSV.name}</span>"

selector_options = [(make_selector_label(i), i) for i in range(len(df_annot))]
dropdown = widgets.Dropdown(options=selector_options, value=0, description="Dog:", layout=widgets.Layout(width="420px"))
btn_prev = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
btn_next = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
btn_unannotated = widgets.Button(description="⏩ Next Unannotated", layout=widgets.Layout(width="160px"))
btn_save = widgets.Button(description="💾 Save CSV", button_style="success", layout=widgets.Layout(width="120px"))
progress_label = widgets.HTML()
status_label = widgets.HTML(value=f"<span style='color:green;'>✓ Loaded {ANNOT_CSV.name}</span>")

dog_info_box = widgets.HTML()
query_crops_box = widgets.HBox()
gallery_crops_box = widgets.HBox()

radio_widgets = {}
for attr in ATTR_CONFIG:
    w = widgets.ToggleButtons(
        options=attr["options"],
        description=f"{attr['label']}:",
        style={"button_width": "90px", "description_width": "150px"},
    )
    radio_widgets[attr["id"]] = w

form_container = widgets.VBox()
table_output = widgets.Output()

def update_ui():
    global current_idx
    row = df_annot.iloc[current_idx]
    d_id = row["Full DOG_ID"]
    
    annotated_cnt = sum(is_dog_annotated(i) for i in range(len(df_annot)))
    pct = int((annotated_cnt / len(df_annot)) * 100)
    progress_label.value = f"<b>Progress:</b> {annotated_cnt}/20 annotated ({pct}%)"
    
    dropdown.unobserve(on_dropdown_change, names="value")
    dropdown.options = [(make_selector_label(i), i) for i in range(len(df_annot))]
    dropdown.value = current_idx
    dropdown.observe(on_dropdown_change, names="value")
    
    grp_badge = f"<span style='background:#38bdf8;color:#000;padding:3px 8px;border-radius:4px;font-weight:bold;'>{row['Group']}</span>" if "Best" in str(row["Group"]) else f"<span style='background:#f97316;color:#fff;padding:3px 8px;border-radius:4px;font-weight:bold;'>{row['Group']}</span>"
    dog_info_box.value = f"""
    <div style='background:#f8fafc;border:1px solid #cbd5e1;padding:12px;border-radius:8px;margin-bottom:10px;'>
      <div style='display:flex;justify-content:space-between;align-items:center;'>
        <div>
          <h3 style='margin:0;font-size:1.15rem;color:#0f172a;'>#{current_idx+1}: {row['Short ID']} {grp_badge}</h3>
          <code style='color:#64748b;font-size:0.8rem;'>{d_id}</code>
        </div>
        <div style='text-align:right;'>
          <span style='margin-right:12px;'>mAP: <b>{row['mAP (%)']:.1f}%</b></span>
          <span style='margin-right:12px;'>Rank-1: <b>{row['Rank-1 (%)']:.1f}%</b></span>
          <span style='margin-right:12px;'>N_query: <b>{row['N_q']}</b></span>
          <span>N_gallery: <b>{row['N_g']}</b></span>
        </div>
      </div>
    </div>
    """
    
    sp = splits[splits["DOG_ID"] == d_id]
    q_vids = sp[sp["SPLIT_CLOSED_SET"] == "query"]["VIDEO_ID"].tolist()
    g_vids = sp[sp["SPLIT_CLOSED_SET"] == "gallery"]["VIDEO_ID"].tolist()
    
    q_items = []
    for i, v in enumerate(q_vids):
        im = load_dog_crop(d_id, v)
        w_img = pil_to_widget_img(im, max_w=170, max_h=170)
        lbl = widgets.HTML(f"<div style='font-size:0.75rem;font-family:monospace;text-align:center;'>Q#{i+1}: {v[:8]}...</div>")
        q_items.append(widgets.VBox([w_img, lbl], layout=widgets.Layout(align_items="center", margin="4px")))
    
    g_items = []
    for i, v in enumerate(g_vids):
        im = load_dog_crop(d_id, v)
        w_img = pil_to_widget_img(im, max_w=170, max_h=170)
        lbl = widgets.HTML(f"<div style='font-size:0.75rem;font-family:monospace;text-align:center;'>G#{i+1}: {v[:8]}...</div>")
        g_items.append(widgets.VBox([w_img, lbl], layout=widgets.Layout(align_items="center", margin="4px")))
    
    query_crops_box.children = q_items
    gallery_crops_box.children = g_items
    
    for attr in ATTR_CONFIG:
        attr_id = attr["id"]
        val = row[attr_id]
        w = radio_widgets[attr_id]
        w.unobserve(on_attr_change, names="value")
        w.value = str(val) if pd.notna(val) and str(val) in attr["options"] else "(Unset)"
        w.observe(on_attr_change, names="value")
        
    with table_output:
        clear_output(wait=True)
        display(df_annot[["Group", "Short ID", "mAP (%)", "Distinctive visual features", "Occlusion", "Blurriness / low resolution", "Lighting", "Viewpoint variation"]])

def on_dropdown_change(change):
    global current_idx
    current_idx = change["new"]
    update_ui()

def on_prev_click(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
        update_ui()

def on_next_click(b):
    global current_idx
    if current_idx < len(df_annot) - 1:
        current_idx += 1
        update_ui()

def on_unannotated_click(b):
    global current_idx
    for i in range(len(df_annot)):
        idx = (current_idx + 1 + i) % len(df_annot)
        if not is_dog_annotated(idx):
            current_idx = idx
            update_ui()
            return
    status_label.value = "<span style='color:blue;font-weight:bold;'>All 20 identities are fully annotated!</span>"

def on_save_click(b):
    save_csv()
    update_ui()

def on_attr_change(change):
    for attr in ATTR_CONFIG:
        attr_id = attr["id"]
        if radio_widgets[attr_id] is change["owner"]:
            new_val = change["new"]
            df_annot.at[current_idx, attr_id] = "" if new_val == "(Unset)" else new_val
            save_csv()
            break
    annotated_cnt = sum(is_dog_annotated(i) for i in range(len(df_annot)))
    pct = int((annotated_cnt / len(df_annot)) * 100)
    progress_label.value = f"<b>Progress:</b> {annotated_cnt}/20 annotated ({pct}%)"
    dropdown.unobserve(on_dropdown_change, names="value")
    dropdown.options = [(make_selector_label(i), i) for i in range(len(df_annot))]
    dropdown.value = current_idx
    dropdown.observe(on_dropdown_change, names="value")
    with table_output:
        clear_output(wait=True)
        display(df_annot[["Group", "Short ID", "mAP (%)", "Distinctive visual features", "Occlusion", "Blurriness / low resolution", "Lighting", "Viewpoint variation"]])

dropdown.observe(on_dropdown_change, names="value")
btn_prev.on_click(on_prev_click)
btn_next.on_click(on_next_click)
btn_unannotated.on_click(on_unannotated_click)
btn_save.on_click(on_save_click)

for attr in ATTR_CONFIG:
    radio_widgets[attr["id"]].observe(on_attr_change, names="value")

form_rows = []
for attr in ATTR_CONFIG:
    w = radio_widgets[attr["id"]]
    desc = widgets.HTML(f"<div style='color:#64748b;font-size:0.8rem;margin-left:150px;margin-bottom:6px;'><em>{attr['desc']}</em></div>")
    form_rows.append(widgets.VBox([w, desc]))
form_container.children = form_rows

nav_box = widgets.HBox([dropdown, btn_prev, btn_next, btn_unannotated, btn_save], layout=widgets.Layout(align_items="center", margin="0 0 10px 0"))
status_box = widgets.HBox([progress_label, widgets.HTML("&nbsp;&nbsp;|&nbsp;&nbsp;"), status_label], layout=widgets.Layout(margin="0 0 10px 0"))

visual_panel = widgets.VBox([
    widgets.HTML("<b>🔍 Query Video Clips:</b>"),
    query_crops_box,
    widgets.HTML("<b style='color:#16a34a;'>🏛 Gallery Video Clips:</b>"),
    gallery_crops_box
], layout=widgets.Layout(border="1px solid #e2e8f0", padding="10px", border_radius="8px", margin="0 0 12px 0"))

annotator_widget = widgets.VBox([
    nav_box,
    status_box,
    dog_info_box,
    visual_panel,
    widgets.HTML("<b>Select Attributes:</b>"),
    form_container,
    widgets.HTML("<hr style='margin:14px 0;'><h4 style='margin:0 0 6px 0;'>Live Annotations Table</h4>"),
    table_output
])

update_ui()
display(annotator_widget)


ModuleNotFoundError: No module named 'ipywidgets'

## 2. Inter-Identity Similarity and Confusion Analysis

**Objective**: Identify dogs that the model frequently confuses with one another and investigate the reasons behind these misidentifications.

### Confusion Types:
1. **Unidirectional Confusion** ($A \to B$): Query videos of Dog $A$ frequently match with gallery videos of Dog $B$ at Rank-1, but queries of Dog $B$ do not retrieve Dog $A$.
2. **Mutual Confusion** ($A \leftrightarrow B$): Queries of Dog $A$ retrieve Dog $B$ at Rank-1, **AND** queries of Dog $B$ retrieve Dog $A$ at Rank-1. Mutual confusion is the strongest indicator that two identities are visually ambiguous or nearly indistinguishable for the visual encoder.

In [ ]:
# =====================================================================
# Compute Unidirectional and Mutual Confusion Pairs
# =====================================================================
unidirectional_counts = Counter()
pair_confusion_stats = defaultdict(lambda: {
    'A_to_B_top1': 0, 'B_to_A_top1': 0,
    'A_to_B_top5': 0, 'B_to_A_top5': 0,
    'examples_A': [], 'examples_B': []
})

for i, q_label in enumerate(query_labels):
    q_item = query_ids[i]
    q_vid = get_vid(q_item)
    ranked_g_labels = gallery_labels[sort_idx[i]]
    ranked_g_items = gallery_ids[sort_idx[i]]
    
    top1_g_label = ranked_g_labels[0]
    top1_g_item = ranked_g_items[0]
    
    # Record Top-1 error
    if top1_g_label != q_label:
        unidirectional_counts[(q_label, top1_g_label)] += 1
        pair = tuple(sorted([q_label, top1_g_label]))
        if q_label == pair[0]:
            pair_confusion_stats[pair]['A_to_B_top1'] += 1
            pair_confusion_stats[pair]['examples_A'].append((q_vid, get_vid(top1_g_item)))
        else:
            pair_confusion_stats[pair]['B_to_A_top1'] += 1
            pair_confusion_stats[pair]['examples_B'].append((q_vid, get_vid(top1_g_item)))
            
    # Record Top-5 errors
    for r in range(5):
        g_lab = ranked_g_labels[r]
        if g_lab != q_label:
            pair = tuple(sorted([q_label, g_lab]))
            if q_label == pair[0]:
                pair_confusion_stats[pair]['A_to_B_top5'] += 1
            else:
                pair_confusion_stats[pair]['B_to_A_top5'] += 1

# 1. Mutual Confusions (Both A->B > 0 and B->A > 0)
mutual_list = []
for pair, data in pair_confusion_stats.items():
    if data['A_to_B_top1'] > 0 and data['B_to_A_top1'] > 0:
        tot_top1 = data['A_to_B_top1'] + data['B_to_A_top1']
        tot_top5 = data['A_to_B_top5'] + data['B_to_A_top5']
        mutual_list.append({
            'Dog A': pair[0],
            'Dog B': pair[1],
            'Short A': pair[0][:8] + '...',
            'Short B': pair[1][:8] + '...',
            'A_to_B (Top-1)': data['A_to_B_top1'],
            'B_to_A (Top-1)': data['B_to_A_top1'],
            'Total Top-1 Mutual': tot_top1,
            'Total Top-5 Mutual': tot_top5,
            'data': data
        })

df_mutual = pd.DataFrame(mutual_list).sort_values(
    by=['Total Top-1 Mutual', 'Total Top-5 Mutual'], ascending=[False, False]
).reset_index(drop=True)

# 2. Top Unidirectional Confusions
unidir_list = []
for (q_id, g_id), cnt in unidirectional_counts.most_common(15):
    unidir_list.append({
        'Query Dog (A)': q_id,
        'False Gallery Match (B)': g_id,
        'Short Query': q_id[:8] + '...',
        'Short False Match': g_id[:8] + '...',
        'Error Count (Top-1)': cnt
    })
df_unidir = pd.DataFrame(unidir_list)

print(f"Found {len(df_mutual)} mutual confusion pairs in the closed-world test set.")
print("\n=== Top Mutual Confusion Pairs (Bidirectional Retrieval Failures) ===")
display(df_mutual[['Short A', 'Short B', 'A_to_B (Top-1)', 'B_to_A (Top-1)', 'Total Top-1 Mutual', 'Total Top-5 Mutual']])

print("\n=== Top Unidirectional Confusions (Query Dog A -> Retrieved Dog B) ===")
display(df_unidir[['Short Query', 'Short False Match', 'Error Count (Top-1)']])

Found 16 mutual confusion pairs in the closed-world test set.

=== Top Mutual Confusion Pairs (Bidirectional Retrieval Failures) ===
        Short A      Short B  ...  Total Top-1 Mutual  Total Top-5 Mutual
0   55f461ae...  e57001df...  ...                   4                   8
1   1aadfd14...  22a635ce...  ...                   3                   5
2   3b1e4347...  92673b1a...  ...                   3                   5
3   77eb8911...  fe7822e8...  ...                   3                   5
4   40989388...  509a29ff...  ...                   2                   8
5   96242e26...  a61b868a...  ...                   2                   8
6   1106673b...  3e600a3f...  ...                   2                   7
7   33c234be...  46cbca2f...  ...                   2                   7
8   89b8b288...  c9dc8631...  ...                   2                   7
9   53303749...  edd2da4e...  ...                   2                   6
10  78ec5c1e...  bb7b36c0...  ...                   2

In [ ]:
# =====================================================================
# Visual Grid: Top Mutual Confusion Pairs
# Triplet: (Query Dog A, False Top-1 Match Dog B, True Gallery Match Dog A)
# =====================================================================
n_display_pairs = min(5, len(df_mutual))
fig, axes = plt.subplots(n_display_pairs, 3, figsize=(15, 4.0 * n_display_pairs), dpi=180)

for p_idx in range(n_display_pairs):
    row = df_mutual.iloc[p_idx]
    d_a = row['Dog A']
    d_b = row['Dog B']
    data = row['data']
    
    # Get query video and false top1 video
    q_vid_a, false_g_vid_b = data['examples_A'][0]
    # Get true gallery video of Dog A
    true_g_vids = [get_vid(i) for i in gallery_ids if get_id(i) == d_a]
    true_g_vid_a = true_g_vids[0] if true_g_vids else None
    
    im_q = load_dog_crop(d_a, q_vid_a)
    im_false = load_dog_crop(d_b, false_g_vid_b)
    im_true = load_dog_crop(d_a, true_g_vid_a)
    
    # Col 1: Query Dog A
    if im_q: axes[p_idx, 0].imshow(im_q)
    axes[p_idx, 0].axis('off')
    axes[p_idx, 0].set_title(f"Pair #{p_idx+1}: Query Dog A\nID: {d_a[:8]}...", fontsize=12, color='#1f77b4', weight='bold')
    
    # Col 2: False Top-1 Match (Dog B)
    if im_false: axes[p_idx, 1].imshow(im_false)
    axes[p_idx, 1].axis('off')
    # Add red outline around false match
    for spine in ['top', 'bottom', 'left', 'right']:
        axes[p_idx, 1].spines[spine].set_visible(True)
        axes[p_idx, 1].spines[spine].set_color('#d62728')
        axes[p_idx, 1].spines[spine].set_linewidth(3.0)
    axes[p_idx, 1].set_title(f"[X] Incorrect Top-1: Dog B\nID: {d_b[:8]}... (Rank-1 Error)", fontsize=12, color='#d62728', weight='bold')
    
    # Col 3: True Gallery Match (Dog A)
    if im_true: axes[p_idx, 2].imshow(im_true)
    axes[p_idx, 2].axis('off')
    for spine in ['top', 'bottom', 'left', 'right']:
        axes[p_idx, 2].spines[spine].set_visible(True)
        axes[p_idx, 2].spines[spine].set_color('#2ca02c')
        axes[p_idx, 2].spines[spine].set_linewidth(3.0)
    axes[p_idx, 2].set_title(f"[OK] Correct Gallery Match: Dog A\nID: {d_a[:8]}...", fontsize=12, color='#2ca02c', weight='bold')

plt.suptitle('Representative Mutual Confusion Triplets:\nQuery Dog A vs. Incorrect Top-1 Match (Dog B) vs. True Gallery Match', fontsize=16, weight='bold', y=0.995)
plt.tight_layout()
plt.savefig(FIG_DIR / 'mutual_confusion_triplets.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'mutual_confusion_triplets.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================================
# Structured Confusion Pairs Comparison Table for User Annotation
# =====================================================================
conf_table_rows = []
for idx, row in df_mutual.head(10).iterrows():
    conf_table_rows.append({
        'Confusion Type': 'Mutual',
        'Dog A (ID)': row['Dog A'],
        'Dog B (ID)': row['Dog B'],
        'Top-1 Count': row['Total Top-1 Mutual'],
        'Top-5 Count': row['Total Top-5 Mutual'],
        'Dog A Breed / Type': '',
        'Dog B Breed / Type': '',
        'Shared Coat Colour': '',
        'Shared Pattern / Markings': '',
        'Shared Silhouette / Ear Shape': '',
        'Key Discriminating Feature Missed': ''
    })

for idx, row in df_unidir.head(10).iterrows():
    conf_table_rows.append({
        'Confusion Type': 'Unidirectional',
        'Dog A (ID)': row['Query Dog (A)'],
        'Dog B (ID)': row['False Gallery Match (B)'],
        'Top-1 Count': row['Error Count (Top-1)'],
        'Top-5 Count': '-',
        'Dog A Breed / Type': '',
        'Dog B Breed / Type': '',
        'Shared Coat Colour': '',
        'Shared Pattern / Markings': '',
        'Shared Silhouette / Ear Shape': '',
        'Key Discriminating Feature Missed': ''
    })

df_conf_annotation = pd.DataFrame(conf_table_rows)
display(df_conf_annotation)

# Save template to CSV
df_conf_annotation.to_csv(ROOT_DIR / "evaluation" / "confusion_pairs_annotation_template.csv", index=False)
print("-> Saved confusion annotation template to evaluation/confusion_pairs_annotation_template.csv")

    Confusion Type  ... Key Discriminating Feature Missed
0           Mutual  ...                                  
1           Mutual  ...                                  
2           Mutual  ...                                  
3           Mutual  ...                                  
4           Mutual  ...                                  
5           Mutual  ...                                  
6           Mutual  ...                                  
7           Mutual  ...                                  
8           Mutual  ...                                  
9           Mutual  ...                                  
10  Unidirectional  ...                                  
11  Unidirectional  ...                                  
12  Unidirectional  ...                                  
13  Unidirectional  ...                                  
14  Unidirectional  ...                                  
15  Unidirectional  ...                                  
16  Unidirecti

## 3. Gallery Size Sensitivity Analysis (Transferred from Ablations)

**Objective**: Quantify how retrieval accuracy scales as the size and composition of the gallery search pool varies.

We present two complementary scaling experiments:
1. **Experiment A (Total Gallery Video Volume Scaling)**: Fixed at all 777 unique gallery identities, varying the volume of distracting videos from $38.9\%$ (1 video per identity) to $100\%$ ($1,998$ videos).
2. **Experiment B (Distractor Identity Scaling)**: Varying the number of unique gallery identities from $N_{\text{IDs}} = 100$ up to $777$ identities, directly measuring how retrieval degrades as more competing individual dogs enter the gallery database.

In [ ]:
# =====================================================================
# Experiment A: Total Gallery Video Volume Scaling (Transferred from Ablations)
# =====================================================================
from evaluation.evaluation_utils import _calc_closed_logic

gallery_steps = [
    {"pct": 38.9, "label": "38.9% (1 vid/ID)", "tick": "38.9%\n(777)", "n_vids": 777, "is_min": True},
    {"pct": 50.0, "label": "50.0%", "tick": "50.0%\n(999)", "n_vids": 999},
    {"pct": 60.0, "label": "60.0%", "tick": "60.0%\n(1,199)", "n_vids": 1199},
    {"pct": 70.0, "label": "70.0%", "tick": "70.0%\n(1,399)", "n_vids": 1399},
    {"pct": 80.0, "label": "80.0%", "tick": "80.0%\n(1,598)", "n_vids": 1598},
    {"pct": 90.0, "label": "90.0%", "tick": "90.0%\n(1,798)", "n_vids": 1798},
    {"pct": 100.0, "label": "100.0% (Full)", "tick": "100.0%\n(1,998)", "n_vids": 1998, "is_full": True},
]

CACHE_FILE_A = ROOT_DIR / "evaluation" / "cache_gallery_sensitivity.pkl"

if CACHE_FILE_A.exists() and CACHE_FILE_A.stat().st_mtime >= baseline_csv.stat().st_mtime:
    with open(CACHE_FILE_A, "rb") as f:
        gallery_summary_steps = pickle.load(f)
    print("-> [CACHE HIT] Loaded gallery volume sensitivity bootstrap results from cache!")
else:
    print("-> Computing 100-iteration bootstrap for gallery volume sensitivity...")
    rng = np.random.RandomState(42)
    N_BOOTSTRAP = 100
    N_GALLERY = len(gallery_labels)
    
    unique_g_pids = np.unique(gallery_labels)
    pid_to_indices = {pid: np.where(gallery_labels == pid)[0] for pid in unique_g_pids}
    
    gallery_summary_steps = []
    for step in gallery_steps:
        target_count = step["n_vids"]
        r1_boots = []
        map_boots = []
        
        for b in range(N_BOOTSTRAP):
            core_idx = [rng.choice(pid_to_indices[pid]) for pid in unique_g_pids]
            core_set = set(core_idx)
            remaining_pool = [i for i in range(N_GALLERY) if i not in core_set]
            
            n_needed = target_count - len(core_idx)
            if n_needed > 0:
                sampled_distractors = rng.choice(remaining_pool, size=n_needed, replace=False)
                sub_g_indices = np.array(core_idx + list(sampled_distractors))
            else:
                sub_g_indices = np.array(core_idx)
                
            sub_dist = dist_mat[:, sub_g_indices]
            sub_g_labels = gallery_labels[sub_g_indices]
            
            match_matrix = query_labels[:, None] == sub_g_labels[None, :]
            sort_idx = np.argsort(sub_dist, axis=1)
            sorted_matches = np.take_along_axis(match_matrix, sort_idx, axis=1)
            
            res = _calc_closed_logic(sorted_matches)
            r1_boots.append(res["cmc"][0] * 100)
            map_boots.append(res["mAP"] * 100)
            
        r1_mean = np.mean(r1_boots)
        r1_ci = (np.percentile(r1_boots, 2.5), np.percentile(r1_boots, 97.5))
        map_mean = np.mean(map_boots)
        map_ci = (np.percentile(map_boots, 2.5), np.percentile(map_boots, 97.5))
        
        gallery_summary_steps.append({
            **step,
            "r1": r1_mean,
            "r1_l": r1_ci[0],
            "r1_u": r1_ci[1],
            "map": map_mean,
            "map_l": map_ci[0],
            "map_u": map_ci[1],
        })
    with open(CACHE_FILE_A, "wb") as f:
        pickle.dump(gallery_summary_steps, f)
    print("-> Cached gallery volume sensitivity results.")

rows_vol = []
for s in gallery_summary_steps:
    rows_vol.append({
        "Gallery Scale": s["label"],
        "Num Videos": s["n_vids"],
        "Rank-1 (%) [95% CI]": f"{s['r1']:.1f} [{s['r1_l']:.1f}-{s['r1_u']:.1f}]",
        "mAP (%) [95% CI]": f"{s['map']:.1f} [{s['map_l']:.1f}-{s['map_u']:.1f}]"
    })
df_sens_vol = pd.DataFrame(rows_vol)
display(df_sens_vol)

-> [CACHE HIT] Loaded gallery volume sensitivity bootstrap results from cache!
      Gallery Scale  Num Videos Rank-1 (%) [95% CI]  mAP (%) [95% CI]
0  38.9% (1 vid/ID)         777    51.6 [49.5-53.6]  62.8 [61.2-64.1]
1             50.0%         999    53.5 [51.7-55.1]  61.2 [59.9-62.3]
2             60.0%        1199    55.5 [54.1-56.9]  60.1 [59.2-61.0]
3             70.0%        1399    57.3 [55.8-58.6]  59.4 [58.5-60.4]
4             80.0%        1598    59.0 [57.9-60.0]  58.8 [58.0-59.3]
5             90.0%        1798    60.6 [59.8-61.3]  58.2 [57.7-58.7]
6     100.0% (Full)        1998    62.1 [62.1-62.1]  57.8 [57.8-57.8]


In [ ]:
# =====================================================================
# Experiment B: Gallery Identity Scaling (Distractor ID Scaling)
# Directly measures degradation as unique competing dog identities increase
# =====================================================================
id_steps = [100, 200, 300, 400, 500, 600, 777]
CACHE_FILE_B = ROOT_DIR / "evaluation" / "cache_gallery_identity_sensitivity.pkl"

if CACHE_FILE_B.exists() and CACHE_FILE_B.stat().st_mtime >= baseline_csv.stat().st_mtime:
    with open(CACHE_FILE_B, "rb") as f:
        id_summary_steps = pickle.load(f)
    print("-> [CACHE HIT] Loaded gallery identity scaling bootstrap results from cache!")
else:
    print("-> Computing 100-iteration bootstrap for gallery identity scaling...")
    rng = np.random.RandomState(42)
    N_BOOTSTRAP = 100
    unique_g_pids = np.unique(gallery_labels)
    
    id_summary_steps = []
    for n_id in id_steps:
        r1_boots = []
        map_boots = []
        
        for b in range(N_BOOTSTRAP):
            if n_id == len(unique_g_pids):
                sampled_pids = set(unique_g_pids)
            else:
                sampled_pids = set(rng.choice(unique_g_pids, size=n_id, replace=False))
                
            sub_g_idx = [i for i, pid in enumerate(gallery_labels) if pid in sampled_pids]
            sub_dist = dist_mat[:, sub_g_idx]
            sub_g_labels = gallery_labels[sub_g_idx]
            
            valid_q_idx = [i for i, qpid in enumerate(query_labels) if qpid in sampled_pids]
            q_sub_dist = sub_dist[valid_q_idx]
            q_sub_labels = query_labels[valid_q_idx]
            
            match_matrix = q_sub_labels[:, None] == sub_g_labels[None, :]
            sort_idx = np.argsort(q_sub_dist, axis=1)
            sorted_matches = np.take_along_axis(match_matrix, sort_idx, axis=1)
            
            res = _calc_closed_logic(sorted_matches)
            r1_boots.append(res["cmc"][0] * 100)
            map_boots.append(res["mAP"] * 100)
            
        r1_mean = np.mean(r1_boots)
        r1_ci = (np.percentile(r1_boots, 2.5), np.percentile(r1_boots, 97.5))
        map_mean = np.mean(map_boots)
        map_ci = (np.percentile(map_boots, 2.5), np.percentile(map_boots, 97.5))
        
        id_summary_steps.append({
            'n_identities': n_id,
            'pct_of_total': (n_id / len(unique_g_pids)) * 100.0,
            'r1': r1_mean, 'r1_l': r1_ci[0], 'r1_u': r1_ci[1],
            'map': map_mean, 'map_l': map_ci[0], 'map_u': map_ci[1]
        })
    with open(CACHE_FILE_B, "wb") as f:
        pickle.dump(id_summary_steps, f)
    print("-> Cached gallery identity scaling results.")

rows_id = []
for s in id_summary_steps:
    rows_id.append({
        "Num Gallery IDs": s["n_identities"],
        "% of Total Population": f"{s['pct_of_total']:.1f}%",
        "Rank-1 (%) [95% CI]": f"{s['r1']:.1f} [{s['r1_l']:.1f}-{s['r1_u']:.1f}]",
        "mAP (%) [95% CI]": f"{s['map']:.1f} [{s['map_l']:.1f}-{s['map_u']:.1f}]"
    })
df_sens_id = pd.DataFrame(rows_id)
display(df_sens_id)

-> [CACHE HIT] Loaded gallery identity scaling bootstrap results from cache!
   Num Gallery IDs % of Total Population Rank-1 (%) [95% CI]  mAP (%) [95% CI]
0              100                 12.9%    82.5 [76.4-89.9]  80.7 [75.3-86.3]
1              200                 25.7%    75.2 [70.2-80.1]  73.1 [69.6-76.8]
2              300                 38.6%    71.6 [67.7-74.8]  68.7 [66.0-71.1]
3              400                 51.5%    68.5 [65.5-71.1]  65.5 [63.5-67.3]
4              500                 64.4%    66.6 [64.3-68.7]  62.9 [61.1-64.8]
5              600                 77.2%    64.5 [62.9-66.4]  60.6 [59.3-62.0]
6              777                100.0%    62.1 [62.1-62.1]  57.8 [57.8-57.8]


In [ ]:
# =====================================================================
# Publication Plot: Side-by-Side Gallery Sensitivity Curves
# Left: Video Volume Scaling | Right: Distractor Identity Scaling
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.0), dpi=200)

# 1. Left Plot: Video Volume Scaling
x_v = np.array([s["pct"] for s in gallery_steps])
labels_v = [s["tick"] for s in gallery_steps]
r1_v = np.array([s["r1"] for s in gallery_summary_steps])
r1_l_v = np.array([s["r1_l"] for s in gallery_summary_steps])
r1_u_v = np.array([s["r1_u"] for s in gallery_summary_steps])
map_v = np.array([s["map"] for s in gallery_summary_steps])
map_l_v = np.array([s["map_l"] for s in gallery_summary_steps])
map_u_v = np.array([s["map_u"] for s in gallery_summary_steps])

ax1.fill_between(x_v, r1_l_v, r1_u_v, color="#1f77b4", alpha=0.15)
ax1.fill_between(x_v, map_l_v, map_u_v, color="#d95f02", alpha=0.15)
ax1.errorbar(x_v, r1_v, yerr=[np.maximum(0, r1_v-r1_l_v), np.maximum(0, r1_u_v-r1_v)],
             fmt="-o", color="#1f77b4", label="Rank-1 (%)", linewidth=2.5, markersize=7.5, capsize=4.5)
ax1.errorbar(x_v, map_v, yerr=[np.maximum(0, map_v-map_l_v), np.maximum(0, map_u_v-map_v)],
             fmt="-s", color="#d95f02", label="mAP (%)", linewidth=2.5, markersize=7.5, capsize=4.5)

for idx in [0, 2, 4, 6]:
    ax1.annotate(f"{r1_v[idx]:.1f}%", xy=(x_v[idx], r1_u_v[idx]), xytext=(0, 6),
                 textcoords="offset points", ha="center", fontsize=11, color="#1f77b4", weight="bold")
    ax1.annotate(f"{map_v[idx]:.1f}%", xy=(x_v[idx], map_l_v[idx]), xytext=(0, -8),
                 textcoords="offset points", ha="center", fontsize=11, color="#b84d00", weight="bold")

ax1.set_xticks(x_v)
ax1.set_xticklabels(labels_v, fontsize=11)
ax1.set_xlabel("Gallery Video Volume (% of Total Tracks)", fontsize=13, labelpad=8)
ax1.set_ylabel("Accuracy (%)", fontsize=13)
ax1.set_title("(A) Video Volume Scaling (Fixed 777 IDs)", fontsize=14, weight="bold", pad=10)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(loc="upper right", framealpha=0.9, fontsize=12)

# 2. Right Plot: Distractor Identity Scaling
x_id = np.array([s["n_identities"] for s in id_summary_steps])
r1_id = np.array([s["r1"] for s in id_summary_steps])
r1_l_id = np.array([s["r1_l"] for s in id_summary_steps])
r1_u_id = np.array([s["r1_u"] for s in id_summary_steps])
map_id = np.array([s["map"] for s in id_summary_steps])
map_l_id = np.array([s["map_l"] for s in id_summary_steps])
map_u_id = np.array([s["map_u"] for s in id_summary_steps])

ax2.fill_between(x_id, r1_l_id, r1_u_id, color="#2ca02c", alpha=0.15)
ax2.fill_between(x_id, map_l_id, map_u_id, color="#7570b3", alpha=0.15)
ax2.errorbar(x_id, r1_id, yerr=[np.maximum(0, r1_id-r1_l_id), np.maximum(0, r1_u_id-r1_id)],
             fmt="-o", color="#2ca02c", label="Rank-1 (%)", linewidth=2.5, markersize=7.5, capsize=4.5)
ax2.errorbar(x_id, map_id, yerr=[np.maximum(0, map_id-map_l_id), np.maximum(0, map_u_id-map_id)],
             fmt="-s", color="#7570b3", label="mAP (%)", linewidth=2.5, markersize=7.5, capsize=4.5)

for idx in [0, 2, 4, 6]:
    ax2.annotate(f"{r1_id[idx]:.1f}%", xy=(x_id[idx], r1_u_id[idx]), xytext=(0, 6),
                 textcoords="offset points", ha="center", fontsize=11, color="#2ca02c", weight="bold")
    ax2.annotate(f"{map_id[idx]:.1f}%", xy=(x_id[idx], map_l_id[idx]), xytext=(0, -8),
                 textcoords="offset points", ha="center", fontsize=11, color="#5e3c99", weight="bold")

ax2.set_xticks(x_id)
ax2.set_xlabel("Number of Competing Gallery Identities (N_IDs)", fontsize=13, labelpad=8)
ax2.set_ylabel("Accuracy (%)", fontsize=13)
ax2.set_title("(B) Distractor Identity Scaling (100 -> 777 IDs)", fontsize=14, weight="bold", pad=10)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(loc="upper right", framealpha=0.9, fontsize=12)

plt.tight_layout()
plt.savefig(FIG_DIR / 'gallery_sensitivity_combined.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'gallery_sensitivity_combined.pdf', bbox_inches='tight')
print(f"-> Saved combined gallery sensitivity figure to {FIG_DIR / 'gallery_sensitivity_combined.png'}")
plt.show()

-> Saved combined gallery sensitivity figure to /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553/evaluation/figures/gallery_sensitivity_combined.png
